# Prefix-Step Accuracy + nterms Generalization Analysis

This notebook does two checks for a trained run:
1. Accuracy at each intermediary prefix step (`term1`, `term2`, ..., final `=`).
2. For variable-length runs (`data.nterms_list`), generalization to **two-under** and **one-over** lengths, with max length gating for non-RoPE transformers.

## Reproducible Run Selection

Choose one of two methods in the next cell:

1. **Direct path (recommended):** set `RUN_REL_PATH` to a repo-relative run directory, e.g.
   - `paper_rerun_models/multi-op/3_term/cot/add_sub`
   - `paper_rerun_models/single-op/4_term/cot/add`

2. **Legacy name lookup:** set `EXPERIMENT_NAME` and leave `RUN_REL_PATH = None`.
   The notebook searches common `experiments/` parent locations.

Checkpoint loading uses `resolve_checkpoint(run_dir)` (prefers best/final based on repo logic).


In [1]:
from pathlib import Path
import random
import numpy as np
import pandas as pd
import torch

# -----------------------------
# Reproducible run selection
# -----------------------------
# Recommended: repo-relative run directory (or absolute path).
RUN_REL_PATH = 'paper_rerun_models/multi-op/3_term/cot/add_sub'
# RUN_REL_PATH = 'paper_rerun_models/single-op/4_term/cot/add'

# Legacy fallback: experiment folder name under experiments roots.
# Used only when RUN_REL_PATH is None.
EXPERIMENT_NAME = None
# EXAMPLE: EXPERIMENT_NAME = 'transformer_modular_addition_subtraction'

# Optional: custom parent dir for EXPERIMENT_NAME lookup (legacy mode).
EXPERIMENTS_BASE_DIR_OVERRIDE = None

# Device override: 'cpu', 'mps', 'cuda', or None to use repo utility default.
DEVICE_OVERRIDE = 'cpu'

# Memory-safety knobs for evaluation.
EVAL_BATCH_SIZE = 163840
EVAL_FORCE_CPU = True

# Number of sampled examples for out-of-distribution nterms (extrapolation/interpolation).
OOD_SAMPLES = 2825761
OOD_SEED = 7

# Generalization evaluation mode: 'sampled', 'exhaustive', or 'hybrid'.
GENERALIZATION_EVAL_MODE = 'hybrid'


In [2]:
import sys

def _find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / 'main.py').exists() and (candidate / 'config').exists():
            return candidate
    raise RuntimeError('Could not locate repo root from current working directory.')

def _resolve_run_dir(repo_root: Path, run_rel_path, experiment_name, base_override):
    # Preferred path-based mode (reproducible across machines).
    if run_rel_path is not None:
        p = Path(run_rel_path)
        if not p.is_absolute():
            p = (repo_root / p).resolve()
        return p, [p.parent]

    # Legacy name-based lookup mode.
    if not experiment_name:
        raise ValueError('Set RUN_REL_PATH or EXPERIMENT_NAME.')

    if base_override is not None:
        base = Path(base_override)
        if not base.is_absolute():
            base = (repo_root / base).resolve()
        return (base / experiment_name).resolve(), [base]

    base_candidates = [
        (repo_root / 'experiments').resolve(),
        (repo_root / 'experiements').resolve(),
        (repo_root.parent / 'experiments').resolve(),
        (repo_root.parent / 'experiements').resolve(),
    ]
    seen = set()
    unique_candidates = []
    for b in base_candidates:
        s = str(b)
        if s not in seen:
            unique_candidates.append(b)
            seen.add(s)

    for base in unique_candidates:
        candidate_run = (base / experiment_name).resolve()
        if candidate_run.exists():
            return candidate_run, unique_candidates

    return (unique_candidates[0] / experiment_name).resolve(), unique_candidates

repo_root = _find_repo_root(Path.cwd())
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from config import Config
from utils import create_dataset, load_model, resolve_checkpoint, get_device
from training.utils import get_sequence_positions

run_dir, checked_bases = _resolve_run_dir(
    repo_root,
    RUN_REL_PATH,
    EXPERIMENT_NAME,
    EXPERIMENTS_BASE_DIR_OVERRIDE,
)
if not run_dir.exists():
    checked_text = '\n'.join([f'  - {str(b)}' for b in checked_bases])
    raise FileNotFoundError(
        f'Run directory not found: {run_dir}.\n'
        f'Checked candidate paths:\n{checked_text}'
    )

config_path = run_dir / 'config.yaml'
if not config_path.exists():
    raise FileNotFoundError(f'config.yaml missing at {config_path}')

config = Config.load(str(config_path), skip_validation=True)
device = get_device(DEVICE_OVERRIDE)

dataset = create_dataset(config.operations, config.data.to_dict(), str(device))
ckpt_path = resolve_checkpoint(str(run_dir))
model, checkpoint = load_model(ckpt_path, config.model.to_dict(), device=str(device))
model.eval()

# Optional safety fallback: move model to CPU for evaluation to avoid MPS OOM.
if EVAL_FORCE_CPU and next(model.parameters()).device.type != 'cpu':
    model = model.to('cpu')
    device = torch.device('cpu')

print('Loaded run:', run_dir)
print('Config:', config_path)
print('Checkpoint:', ckpt_path)
print('Model type:', config.model_type)
print('Operations:', config.operations)
print('training_target:', config.training.training_target)
print('nterms:', config.data.nterms)
print('nterms_list:', config.data.nterms_list)
print('use_rope:', getattr(config.model, 'use_rope', False))
print('max_len:', getattr(config.model, 'max_len', None))
print('device:', device)


Loaded run: /Users/adirathodd/Desktop/NJIT/rnn-research/icml_submissions/cramming/paper_rerun_models/multi-op/3_term/cot/add_sub
Config: /Users/adirathodd/Desktop/NJIT/rnn-research/icml_submissions/cramming/paper_rerun_models/multi-op/3_term/cot/add_sub/config.yaml
Checkpoint: /Users/adirathodd/Desktop/NJIT/rnn-research/icml_submissions/cramming/paper_rerun_models/multi-op/3_term/cot/add_sub/checkpoints/best.pt
Model type: transformer
Operations: ['addition', 'subtraction']
training_target: seq_cot
nterms: 3
nterms_list: None
use_rope: True
max_len: 6
device: cpu


In [3]:
def _op_token_maps(ds):
    includes_zero = bool(ds.includes_zero)
    modulo = int(ds.modulo)
    equals_token = modulo if includes_zero else (modulo - 1)
    op_to_token = {op: equals_token + 1 + i for i, op in enumerate(ds.operations)}
    token_to_op = {tok: op for op, tok in op_to_token.items()}
    return equals_token, op_to_token, token_to_op

def _decode_term_token(tok, includes_zero):
    tok_int = int(tok)
    return tok_int if includes_zero else (tok_int + 1)

def _encode_result_value(val, includes_zero):
    return int(val) if includes_zero else int(val) - 1

def _predict_sequence_logits_batched(model, data, batch_size):
    device = next(model.parameters()).device
    logits_parts = []
    with torch.inference_mode():
        for start in range(0, int(data.shape[0]), int(batch_size)):
            end = min(start + int(batch_size), int(data.shape[0]))
            batch = data[start:end].to(device)
            logits_parts.append(model(batch, return_sequence_logits=True).cpu())
    return torch.cat(logits_parts, dim=0) if logits_parts else torch.empty((0, data.shape[1], model.vocab_size))

def _predict_final_logits_batched(model, data, batch_size):
    device = next(model.parameters()).device
    logits_parts = []
    with torch.inference_mode():
        for start in range(0, int(data.shape[0]), int(batch_size)):
            end = min(start + int(batch_size), int(data.shape[0]))
            batch = data[start:end].to(device)
            logits_parts.append(model(batch).cpu())
    return torch.cat(logits_parts, dim=0) if logits_parts else torch.empty((0, model.vocab_size))

def build_sequence_targets_from_tokens(tokens, ds, nterms=None, ignore_index=-100):
    if not isinstance(tokens, torch.Tensor):
        tokens = torch.as_tensor(tokens, dtype=torch.long)
    tokens_cpu = tokens.detach().cpu()

    if nterms is None:
        nterms = tokens_cpu.shape[1] // 2

    positions = get_sequence_positions(int(nterms))
    term_positions = positions['term_positions']
    op_positions = positions['op_positions']
    equals_pos = int(positions['equals_position'])

    modulo = int(ds.modulo)
    includes_zero = bool(ds.includes_zero)
    _, _, token_to_op = _op_token_maps(ds)

    out = torch.full((tokens_cpu.shape[0], tokens_cpu.shape[1]), ignore_index, dtype=torch.long)

    for b in range(tokens_cpu.shape[0]):
        seq = tokens_cpu[b]
        running = _decode_term_token(seq[term_positions[0]], includes_zero)
        out[b, term_positions[0]] = _encode_result_value(running, includes_zero)

        for step_idx in range(1, int(nterms)):
            op_tok = int(seq[op_positions[step_idx - 1]])
            op_name = token_to_op[op_tok]
            rhs = _decode_term_token(seq[term_positions[step_idx]], includes_zero)

            if op_name == 'addition':
                running = (running + rhs) % modulo
            elif op_name == 'subtraction':
                running = (running - rhs) % modulo
            elif op_name == 'multiplication':
                running = (running * rhs) % modulo
            elif op_name == 'division':
                running = (running * pow(rhs, -1, modulo)) % modulo
            else:
                raise ValueError(f'Unsupported operation: {op_name}')

            out[b, term_positions[step_idx]] = _encode_result_value(running, includes_zero)

        out[b, equals_pos] = _encode_result_value(running, includes_zero)

    return out

def _sample_varlen_batch(ds, nterms, num_samples, seed=0):
    rng = random.Random(seed)
    modulo = int(ds.modulo)
    includes_zero = bool(ds.includes_zero)
    operations = list(ds.operations)
    mixed_sequence_ops = bool(getattr(ds, 'mixed_sequence_operations', False))

    equals_token, op_to_token, _ = _op_token_maps(ds)
    valid_divisors = [x for x in range(1, modulo) if np.gcd(x, modulo) == 1]

    seqs = []
    for _ in range(int(num_samples)):
        if len(operations) == 1:
            op_pattern = [operations[0]] * (nterms - 1)
        elif mixed_sequence_ops:
            op_pattern = [rng.choice(operations) for _ in range(nterms - 1)]
        else:
            op = rng.choice(operations)
            op_pattern = [op] * (nterms - 1)

        term_vals = []
        if includes_zero:
            term_vals.append(rng.randrange(0, modulo))
        else:
            term_vals.append(rng.randrange(1, modulo))

        for op in op_pattern:
            if op == 'division':
                term_vals.append(rng.choice(valid_divisors))
            else:
                if includes_zero:
                    term_vals.append(rng.randrange(0, modulo))
                else:
                    term_vals.append(rng.randrange(1, modulo))

        seq = torch.empty((2 * nterms,), dtype=torch.long)
        for i in range(nterms):
            seq[2 * i] = term_vals[i] if includes_zero else (term_vals[i] - 1)
            if i < nterms - 1:
                seq[2 * i + 1] = op_to_token[op_pattern[i]]
        seq[-1] = equals_token
        seqs.append(seq)

    data = torch.stack(seqs, dim=0)
    labels_seq = build_sequence_targets_from_tokens(data, ds, nterms=nterms)
    final_labels = labels_seq[:, -1]
    return data, final_labels, labels_seq

def evaluate_prefix_position_accuracy(model, data, labels_seq, nterms, batch_size=EVAL_BATCH_SIZE):
    labels_cpu = labels_seq.detach().cpu()
    logits_cpu = _predict_sequence_logits_batched(model, data, batch_size=batch_size)
    pred_cpu = logits_cpu.argmax(dim=-1)

    pos = get_sequence_positions(int(nterms))
    term_positions = pos['term_positions']
    equals_pos = int(pos['equals_position'])

    rows = []
    for i, p in enumerate(term_positions, start=1):
        mask = labels_cpu[:, p] >= 0
        total = int(mask.sum().item())
        correct = int((pred_cpu[:, p][mask] == labels_cpu[:, p][mask]).sum().item()) if total else 0
        acc = 100.0 * correct / total if total else 0.0
        rows.append({
            'position_name': f'term{i}',
            'position_index': int(p),
            'accuracy': acc,
            'correct': correct,
            'total': total,
        })

    mask = labels_cpu[:, equals_pos] >= 0
    total = int(mask.sum().item())
    correct = int((pred_cpu[:, equals_pos][mask] == labels_cpu[:, equals_pos][mask]).sum().item()) if total else 0
    acc = 100.0 * correct / total if total else 0.0
    rows.append({
        'position_name': 'final_equals',
        'position_index': equals_pos,
        'accuracy': acc,
        'correct': correct,
        'total': total,
    })

    return rows

def evaluate_final_accuracy(model, data, final_labels, batch_size=EVAL_BATCH_SIZE):
    labels_cpu = final_labels.detach().cpu()
    logits_cpu = _predict_final_logits_batched(model, data, batch_size=batch_size)
    pred_cpu = logits_cpu.argmax(dim=-1)
    total = int(labels_cpu.numel())
    correct = int((pred_cpu == labels_cpu).sum().item())
    return (100.0 * correct / total) if total else 0.0

def enforce_length_constraint_for_model(cfg, nterms):
    if cfg.model_type != 'transformer':
        return True, ''
    use_rope = bool(getattr(cfg.model, 'use_rope', False))
    if use_rope:
        return True, ''

    max_len = int(getattr(cfg.model, 'max_len', 0))
    seq_len = 2 * int(nterms)
    if seq_len <= max_len:
        return True, ''

    return False, f'skipped: seq_len={seq_len} exceeds non-RoPE max_len={max_len}'

## 1) Prefix-step accuracy at each intermediary position

This reports accuracy at `term1`, `term2`, ..., and final `=` position.

In [4]:
prefix_rows = []

if getattr(dataset, 'is_variable_length', False):
    # Use full in-distribution data by concatenating train + test per nterms.
    train_splits_seq = dataset.get_train_splits(target='seq')
    test_splits_seq = dataset.get_test_splits(target='seq')
    all_nterms = sorted(set(train_splits_seq.keys()) | set(test_splits_seq.keys()))

    for nterms in all_nterms:
        parts_data, parts_labels = [], []
        if nterms in train_splits_seq:
            d_tr, y_tr = train_splits_seq[nterms]
            parts_data.append(d_tr)
            parts_labels.append(y_tr)
        if nterms in test_splits_seq:
            d_te, y_te = test_splits_seq[nterms]
            parts_data.append(d_te)
            parts_labels.append(y_te)

        if not parts_data:
            continue

        data_n = torch.cat(parts_data, dim=0)
        labels_seq_n = torch.cat(parts_labels, dim=0)

        rows = evaluate_prefix_position_accuracy(
            model, data_n, labels_seq_n, nterms=int(nterms), batch_size=EVAL_BATCH_SIZE
        )
        for row in rows:
            row['nterms'] = int(nterms)
            row['source'] = 'all_split_seq'
            prefix_rows.append(row)
else:
    nterms = int(config.data.nterms)
    full_data_operation = 'all' if 'all' in getattr(dataset, 'operations', []) else dataset.operations[0]
    # Fast path for fixed-length datasets: fetch seq labels directly.
    try:
        data_all, labels_seq = dataset.get_full_data(operation=full_data_operation, target='seq')
        source_tag = 'all_split_seq'
    except Exception:
        # Fallback for datasets without seq targets exposed.
        data_all = dataset.get_full_data(operation=full_data_operation, target='final')[0]
        labels_seq = build_sequence_targets_from_tokens(data_all, dataset, nterms=nterms)
        source_tag = 'rebuilt_seq'

    rows = evaluate_prefix_position_accuracy(
        model, data_all, labels_seq, nterms=nterms, batch_size=EVAL_BATCH_SIZE
    )
    for row in rows:
        row['nterms'] = nterms
        row['source'] = source_tag
        prefix_rows.append(row)

prefix_df = pd.DataFrame(prefix_rows).sort_values(['nterms', 'position_index']).reset_index(drop=True)
display(prefix_df)

,position_name,position_index,accuracy,correct,total,nterms,source
0,term1,0,100.0,68921,68921,3,all_split_seq
1,term2,2,100.0,68921,68921,3,all_split_seq
2,term3,4,100.0,68921,68921,3,all_split_seq
3,final_equals,5,100.0,68921,68921,3,all_split_seq


## 1.1) Prefix accuracy by operation sequence pattern

This creates separate tables for each operation pattern (for example `++`, `--`, `+-`, `-+` for add/sub with `nterms=3`).

In [5]:
def _pattern_symbol(op_name):
    symbols = {
        'addition': '+',
        'subtraction': '-',
        'multiplication': '*',
        'division': '/',
    }
    return symbols.get(op_name, op_name[:1])

def _extract_pattern_strings(data, ds, nterms):
    positions = get_sequence_positions(int(nterms))
    op_positions = positions['op_positions']
    _, _, token_to_op = _op_token_maps(ds)

    data_cpu = data.detach().cpu()
    patterns = []
    for row in data_cpu:
        ops = []
        for pos in op_positions:
            op_name = token_to_op[int(row[pos])]
            ops.append(_pattern_symbol(op_name))
        patterns.append(''.join(ops))
    return patterns

def _make_pattern_tables_for_split(model, ds, data, nterms, labels_seq=None):
    if labels_seq is None:
        labels_seq = build_sequence_targets_from_tokens(data, ds, nterms=int(nterms))
    patterns = _extract_pattern_strings(data, ds, nterms=int(nterms))

    tables = {}
    unique_patterns = sorted(set(patterns))
    for pat in unique_patterns:
        idx = [i for i, p in enumerate(patterns) if p == pat]
        if not idx:
            continue

        idx_t = torch.tensor(idx, dtype=torch.long)
        data_pat = data[idx_t]
        labels_pat = labels_seq[idx_t]

        rows = evaluate_prefix_position_accuracy(
            model,
            data_pat,
            labels_pat,
            nterms=int(nterms),
            batch_size=EVAL_BATCH_SIZE,
        )
        df_pat = pd.DataFrame(rows).sort_values('position_index').reset_index(drop=True)
        df_pat['nterms'] = int(nterms)
        df_pat['pattern'] = pat
        df_pat['num_samples'] = int(len(idx))
        tables[(int(nterms), pat)] = df_pat

    return tables

pattern_tables = {}

if getattr(dataset, 'is_variable_length', False):
    try:
        train_splits_seq = dataset.get_train_splits(target='seq')
        test_splits_seq = dataset.get_test_splits(target='seq')
        all_nterms = sorted(set(train_splits_seq.keys()) | set(test_splits_seq.keys()))

        for nterms in all_nterms:
            parts_data, parts_labels = [], []
            if nterms in train_splits_seq:
                d_tr, y_tr = train_splits_seq[nterms]
                parts_data.append(d_tr)
                parts_labels.append(y_tr)
            if nterms in test_splits_seq:
                d_te, y_te = test_splits_seq[nterms]
                parts_data.append(d_te)
                parts_labels.append(y_te)
            if not parts_data:
                continue

            data_n = torch.cat(parts_data, dim=0)
            labels_seq_n = torch.cat(parts_labels, dim=0)
            pattern_tables.update(
                _make_pattern_tables_for_split(
                    model,
                    dataset,
                    data_n,
                    nterms=int(nterms),
                    labels_seq=labels_seq_n,
                )
            )
    except Exception:
        train_splits = dataset.get_train_splits(target='final')
        test_splits = dataset.get_test_splits(target='final')
        all_nterms = sorted(set(train_splits.keys()) | set(test_splits.keys()))
        for nterms in all_nterms:
            parts = []
            if nterms in train_splits:
                parts.append(train_splits[nterms][0])
            if nterms in test_splits:
                parts.append(test_splits[nterms][0])
            if not parts:
                continue
            data_n = torch.cat(parts, dim=0)
            pattern_tables.update(_make_pattern_tables_for_split(model, dataset, data_n, nterms=int(nterms)))
else:
    nterms = int(config.data.nterms)
    full_data_operation = 'all' if 'all' in getattr(dataset, 'operations', []) else dataset.operations[0]
    try:
        data_all, labels_seq_all = dataset.get_full_data(operation=full_data_operation, target='seq')
    except Exception:
        data_all = dataset.get_full_data(operation=full_data_operation, target='final')[0]
        labels_seq_all = None
    pattern_tables.update(
        _make_pattern_tables_for_split(
            model,
            dataset,
            data_all,
            nterms=nterms,
            labels_seq=labels_seq_all,
        )
    )

if not pattern_tables:
    print('No pattern tables to display.')
else:
    for key in sorted(pattern_tables.keys()):
        nterms, pattern = key
        print(f'nterms={nterms} | pattern={pattern}')
        display(pattern_tables[key])

nterms=3 | pattern=++


,position_name,position_index,accuracy,correct,total,nterms,pattern,num_samples
0,term1,0,100.0,68921,68921,3,++,68921
1,term2,2,100.0,68921,68921,3,++,68921
2,term3,4,100.0,68921,68921,3,++,68921
3,final_equals,5,100.0,68921,68921,3,++,68921


## 1.2) Prefix accuracy on intermixed operation patterns

This evaluates **intermixed** operator sequences only (for example `+-` and `-+` when operations are add/sub and `nterms=3`).


In [6]:
import itertools
import math
import random

INTERMIX_MAX_SAMPLES_PER_PATTERN = int(globals().get('INTERMIX_MAX_SAMPLES_PER_PATTERN', 200000))
INTERMIX_SEED = int(globals().get('INTERMIX_SEED', 1234))

def _intermixed_patterns(operations, nterms):
    ops = list(operations)
    if len(ops) < 2 or int(nterms) < 3:
        return []
    all_patterns = itertools.product(ops, repeat=int(nterms) - 1)
    return [pat for pat in all_patterns if len(set(pat)) > 1]

def _term_domain(ds, op_name):
    modulo = int(ds.modulo)
    if op_name == 'division':
        return [x for x in range(1, modulo) if math.gcd(x, modulo) == 1]
    if bool(ds.includes_zero):
        return list(range(0, modulo))
    return list(range(1, modulo))

def _build_mixed_batch(ds, op_pattern, terms_batch):
    includes_zero = bool(ds.includes_zero)
    include_equals_token = bool(getattr(ds, 'include_equals_token', True))
    modulo = int(ds.modulo)
    equals_token, op_to_token, _ = _op_token_maps(ds)

    nterms = len(op_pattern) + 1
    terms_t = torch.as_tensor(terms_batch, dtype=torch.long)
    batch_size_local = int(terms_t.shape[0])

    seq_len = (2 * nterms) if include_equals_token else (2 * nterms - 1)
    data = torch.empty((batch_size_local, seq_len), dtype=torch.long)
    data[:, 0:seq_len:2] = terms_t if includes_zero else (terms_t - 1)

    if nterms > 1:
        op_tokens = torch.tensor([op_to_token[op] for op in op_pattern], dtype=torch.long)
        data[:, 1:seq_len-1:2] = op_tokens.unsqueeze(0).expand(batch_size_local, -1)

    if include_equals_token:
        data[:, -1] = int(equals_token)

    labels_seq = torch.full((batch_size_local, seq_len), -100, dtype=torch.long)
    running = terms_t[:, 0].clone()
    labels_seq[:, 0] = running if includes_zero else (running - 1)

    for step_idx in range(1, nterms):
        rhs = terms_t[:, step_idx]
        op_name = op_pattern[step_idx - 1]
        if op_name == 'addition':
            running = (running + rhs) % modulo
        elif op_name == 'subtraction':
            running = (running - rhs) % modulo
        elif op_name == 'multiplication':
            running = (running * rhs) % modulo
        elif op_name == 'division':
            inv_rhs = torch.as_tensor([pow(int(v), -1, modulo) for v in rhs.tolist()], dtype=torch.long)
            running = (running * inv_rhs) % modulo
        else:
            raise ValueError(f'Unsupported operation: {op_name}')

        out_pos = 2 * step_idx
        if out_pos < seq_len:
            labels_seq[:, out_pos] = running if includes_zero else (running - 1)

    labels_seq[:, -1] = running if includes_zero else (running - 1)
    return data, labels_seq

def _pattern_dataset_size(ds, op_pattern):
    sizes = [_term_domain(ds, 'first_term')]
    for op_name in op_pattern:
        sizes.append(_term_domain(ds, op_name))
    total = 1
    for dom in sizes:
        total *= len(dom)
    return int(total)

def _sample_terms_for_pattern(ds, op_pattern, num_samples, seed):
    rng = random.Random(int(seed))
    domains = [_term_domain(ds, 'first_term')]
    for op_name in op_pattern:
        domains.append(_term_domain(ds, op_name))

    out = []
    for _ in range(int(num_samples)):
        out.append(tuple(rng.choice(dom) for dom in domains))
    return out

nterms_eval = int(config.data.nterms)
mixed_patterns = _intermixed_patterns(dataset.operations, nterms_eval)

intermixed_rows = []
intermixed_tables = {}

if not mixed_patterns:
    print('No intermixed patterns available (need >=2 operations and nterms >= 3).')
else:
    for pat_idx, pat in enumerate(mixed_patterns):
        size = _pattern_dataset_size(dataset, pat)
        use_samples = min(int(INTERMIX_MAX_SAMPLES_PER_PATTERN), int(size))
        exhaustive = (use_samples == size)

        if exhaustive:
            domains = [_term_domain(dataset, 'first_term')]
            for op_name in pat:
                domains.append(_term_domain(dataset, op_name))
            terms = list(itertools.product(*domains))
        else:
            terms = _sample_terms_for_pattern(
                dataset,
                pat,
                num_samples=use_samples,
                seed=INTERMIX_SEED + int(pat_idx),
            )

        data_pat, labels_seq_pat = _build_mixed_batch(dataset, pat, terms)
        rows = evaluate_prefix_position_accuracy(
            model,
            data_pat,
            labels_seq_pat,
            nterms=nterms_eval,
            batch_size=EVAL_BATCH_SIZE,
        )

        pat_str = ''.join(_pattern_symbol(op) for op in pat)
        df_pat = pd.DataFrame(rows).sort_values('position_index').reset_index(drop=True)
        df_pat['nterms'] = int(nterms_eval)
        df_pat['pattern'] = pat_str
        df_pat['num_samples'] = int(len(terms))
        df_pat['eval_mode'] = 'exhaustive' if exhaustive else 'sampled'
        intermixed_tables[pat_str] = df_pat

        row_map = {r['position_name']: r for r in rows}
        step_acc = {}
        for i in range(1, int(nterms_eval) + 1):
            letter = chr(ord('a') + i - 1)
            term_row = row_map.get(f'term{i}')
            step_acc[letter] = float(term_row['accuracy']) if term_row is not None else float('nan')
        eq_row = row_map.get('final_equals')
        step_acc['='] = float(eq_row['accuracy']) if eq_row is not None else float('nan')

        intermixed_rows.append({
            'nterms': int(nterms_eval),
            'pattern': pat_str,
            'num_samples': int(len(terms)),
            'dataset_size_for_pattern': int(size),
            'eval_mode': 'exhaustive' if exhaustive else 'sampled',
            **step_acc,
            'final_accuracy': step_acc['='],
        })

    intermixed_summary_df = pd.DataFrame(intermixed_rows).sort_values(['nterms', 'pattern']).reset_index(drop=True)
    display(intermixed_summary_df)

    base_cols = ['nterms', 'pattern', 'num_samples', 'eval_mode']
    step_cols = [chr(ord('a') + i) for i in range(int(nterms_eval))] + ['=']
    compact_cols = [c for c in (base_cols + step_cols) if c in intermixed_summary_df.columns]
    print('Intermixed step accuracy (a, b, c, =):')
    display(intermixed_summary_df[compact_cols])

    for pat in sorted(intermixed_tables.keys()):
        print(f'nterms={nterms_eval} | intermixed pattern={pat}')
        display(intermixed_tables[pat])



,nterms,pattern,num_samples,dataset_size_for_pattern,eval_mode,a,b,c,=,final_accuracy
0,3,+-,68921,68921,exhaustive,100.0,100.0,2.450632,2.460788,2.460788
1,3,-+,68921,68921,exhaustive,100.0,100.0,2.610235,2.708899,2.708899


Intermixed step accuracy (a, b, c, =):


,nterms,pattern,num_samples,eval_mode,a,b,c,=
0,3,+-,68921,exhaustive,100.0,100.0,2.450632,2.460788
1,3,-+,68921,exhaustive,100.0,100.0,2.610235,2.708899


nterms=3 | intermixed pattern=+-


,position_name,position_index,accuracy,correct,total,nterms,pattern,num_samples,eval_mode
0,term1,0,100.000000,68921,68921,3,+-,68921,exhaustive
1,term2,2,100.000000,68921,68921,3,+-,68921,exhaustive
2,term3,4,2.450632,1689,68921,3,+-,68921,exhaustive
3,final_equals,5,2.460788,1696,68921,3,+-,68921,exhaustive


nterms=3 | intermixed pattern=-+


,position_name,position_index,accuracy,correct,total,nterms,pattern,num_samples,eval_mode
0,term1,0,100.000000,68921,68921,3,-+,68921,exhaustive
1,term2,2,100.000000,68921,68921,3,-+,68921,exhaustive
2,term3,4,2.610235,1799,68921,3,-+,68921,exhaustive
3,final_equals,5,2.708899,1867,68921,3,-+,68921,exhaustive


## 2) nterms generalization: two-under and one-over

If `data.nterms_list` exists, this evaluates:
- two-under: `min(nterms_list)-2` and `min(nterms_list)-1` (valid lengths only, `>=2`)
- one-over: `max(nterms_list)+1`

For non-RoPE transformers, candidates with `2*nterms > max_len` are skipped.

In [ ]:
import itertools
import math

def _iter_op_patterns(ds, nterms):
    operations = list(ds.operations)
    mixed_sequence_ops = bool(getattr(ds, 'mixed_sequence_operations', False))
    if len(operations) == 1:
        yield tuple([operations[0]] * (nterms - 1))
    elif mixed_sequence_ops:
        for pat in itertools.product(operations, repeat=nterms - 1):
            yield tuple(pat)
    else:
        for op in operations:
            yield tuple([op] * (nterms - 1))

def _term_domain_for_op(ds, op_name):
    if op_name == 'division':
        return [x for x in range(1, int(ds.modulo)) if math.gcd(x, int(ds.modulo)) == 1]
    if bool(ds.includes_zero):
        return list(range(0, int(ds.modulo)))
    return list(range(1, int(ds.modulo)))

def _build_batch_from_terms(ds, op_pattern, terms_batch):
    includes_zero = bool(ds.includes_zero)
    modulo = int(ds.modulo)
    equals_token, op_to_token, _ = _op_token_maps(ds)
    nterms = len(op_pattern) + 1

    terms_t = torch.as_tensor(terms_batch, dtype=torch.long)
    batch_size_local = int(terms_t.shape[0])

    data = torch.empty((batch_size_local, 2 * nterms), dtype=torch.long)
    data[:, 0::2] = terms_t if includes_zero else (terms_t - 1)
    if nterms > 1:
        op_tokens = torch.tensor([op_to_token[op] for op in op_pattern], dtype=torch.long)
        data[:, 1:-1:2] = op_tokens.unsqueeze(0).expand(batch_size_local, -1)
    data[:, -1] = equals_token

    labels_seq = torch.full((batch_size_local, 2 * nterms), -100, dtype=torch.long)
    running = terms_t[:, 0].clone()
    labels_seq[:, 0] = running if includes_zero else (running - 1)

    for step_idx in range(1, nterms):
        rhs = terms_t[:, step_idx]
        op_name = op_pattern[step_idx - 1]
        if op_name == 'addition':
            running = (running + rhs) % modulo
        elif op_name == 'subtraction':
            running = (running - rhs) % modulo
        elif op_name == 'multiplication':
            running = (running * rhs) % modulo
        elif op_name == 'division':
            inv_rhs = torch.as_tensor([pow(int(v), -1, modulo) for v in rhs.tolist()], dtype=torch.long)
            running = (running * inv_rhs) % modulo
        else:
            raise ValueError(f'Unsupported operation: {op_name}')

        out_pos = 2 * step_idx
        labels_seq[:, out_pos] = running if includes_zero else (running - 1)

    labels_seq[:, -1] = running if includes_zero else (running - 1)
    return data, labels_seq

def _count_total_exhaustive_samples(ds, nterms):
    total = 0
    first_domain_size = int(ds.modulo) if bool(ds.includes_zero) else int(ds.modulo) - 1
    for op_pattern in _iter_op_patterns(ds, nterms):
        size = first_domain_size
        for op_name in op_pattern:
            if op_name == 'division':
                size *= sum(1 for x in range(1, int(ds.modulo)) if math.gcd(x, int(ds.modulo)) == 1)
            else:
                size *= first_domain_size
        total += size
    return int(total)

def evaluate_exhaustive_nterms_batched(model, ds, nterms, batch_size=EVAL_BATCH_SIZE):
    nterms = int(nterms)
    device = next(model.parameters()).device
    positions = get_sequence_positions(nterms)
    term_positions = positions['term_positions']
    equals_pos = int(positions['equals_position'])

    total_samples = 0
    final_correct = 0
    term_correct = {int(p): 0 for p in term_positions}

    for op_pattern in _iter_op_patterns(ds, nterms):
        domains = []
        first_domain = _term_domain_for_op(ds, 'first_term')
        domains.append(first_domain)
        for op_name in op_pattern:
            domains.append(_term_domain_for_op(ds, op_name))

        terms_buffer = []
        for terms in itertools.product(*domains):
            terms_buffer.append(terms)
            if len(terms_buffer) >= int(batch_size):
                data_b, labels_seq_b = _build_batch_from_terms(ds, op_pattern, terms_buffer)
                terms_buffer = []

                with torch.inference_mode():
                    logits_seq = model(data_b.to(device), return_sequence_logits=True).cpu()
                pred = logits_seq.argmax(dim=-1)

                final_correct += int((pred[:, equals_pos] == labels_seq_b[:, equals_pos]).sum().item())
                total_samples += int(labels_seq_b.shape[0])

                for p in term_positions:
                    term_correct[int(p)] += int((pred[:, p] == labels_seq_b[:, p]).sum().item())

        if terms_buffer:
            data_b, labels_seq_b = _build_batch_from_terms(ds, op_pattern, terms_buffer)
            with torch.inference_mode():
                logits_seq = model(data_b.to(device), return_sequence_logits=True).cpu()
            pred = logits_seq.argmax(dim=-1)

            final_correct += int((pred[:, equals_pos] == labels_seq_b[:, equals_pos]).sum().item())
            total_samples += int(labels_seq_b.shape[0])

            for p in term_positions:
                term_correct[int(p)] += int((pred[:, p] == labels_seq_b[:, p]).sum().item())

    final_accuracy = (100.0 * final_correct / total_samples) if total_samples else float('nan')
    per_term_accuracy = {}
    for i, p in enumerate(term_positions, start=1):
        per_term_accuracy[f'term{i}_accuracy'] = (
            100.0 * term_correct[int(p)] / total_samples
        ) if total_samples else float('nan')

    return {
        'num_samples': int(total_samples),
        'final_accuracy': float(final_accuracy),
        'per_term_accuracy': per_term_accuracy,
    }

def evaluate_sampled_nterms(model, ds, nterms, num_samples, seed, batch_size=EVAL_BATCH_SIZE):
    data, _, labels_seq = _sample_varlen_batch(ds, nterms=int(nterms), num_samples=int(num_samples), seed=int(seed))
    rows = evaluate_prefix_position_accuracy(
        model,
        data,
        labels_seq,
        nterms=int(nterms),
        batch_size=batch_size,
    )

    final_row = next((r for r in rows if r['position_name'] == 'final_equals'), None)
    per_term_accuracy = {}
    for i in range(1, int(nterms) + 1):
        term_row = next((r for r in rows if r['position_name'] == f'term{i}'), None)
        per_term_accuracy[f'term{i}_accuracy'] = float(term_row['accuracy']) if term_row is not None else float('nan')

    return {
        'num_samples': int(data.shape[0]),
        'final_accuracy': float(final_row['accuracy']) if final_row is not None else float('nan'),
        'per_term_accuracy': per_term_accuracy,
    }

def _reuse_in_distribution_stats_from_prefix_df(nterms):
    cached = globals().get('prefix_df', None)
    if not isinstance(cached, pd.DataFrame) or cached.empty:
        return None

    needed_cols = {'nterms', 'position_name', 'accuracy'}
    if not needed_cols.issubset(set(cached.columns)):
        return None

    nterms_series = pd.to_numeric(cached['nterms'], errors='coerce')
    rows_n = cached[nterms_series == int(nterms)]
    if rows_n.empty:
        return None

    final_row = rows_n[rows_n['position_name'] == 'final_equals']
    if final_row.empty:
        return None

    per_term_accuracy = {}
    for i in range(1, int(nterms) + 1):
        term_row = rows_n[rows_n['position_name'] == f'term{i}']
        if term_row.empty:
            return None
        per_term_accuracy[f'term{i}_accuracy'] = float(term_row.iloc[0]['accuracy'])

    if 'total' in final_row.columns:
        num_samples = int(final_row.iloc[0]['total'])
    elif 'num_samples' in final_row.columns:
        num_samples = int(final_row.iloc[0]['num_samples'])
    else:
        num_samples = 0

    return {
        'num_samples': num_samples,
        'final_accuracy': float(final_row.iloc[0]['accuracy']),
        'per_term_accuracy': per_term_accuracy,
    }

base_nterms = int(config.data.nterms)
eval_lengths = sorted(set([base_nterms - 1, base_nterms, base_nterms + 1]))

generalization_rows = []
max_eval_nterms = max(eval_lengths) if eval_lengths else base_nterms

def _dataset_size_for_nterms(ds, nterms):
    nterms = int(nterms)
    if bool(getattr(ds, 'is_variable_length', False)):
        try:
            test_splits = ds.get_test_splits(target='final')
            if nterms in test_splits:
                return int(test_splits[nterms][0].shape[0])
        except Exception:
            pass
        # Fall back to full combinatorial size when cached split is unavailable.
        return int(_count_total_exhaustive_samples(ds, nterms))

    try:
        base_n = int(config.data.nterms)
        if nterms == base_n:
            data_test = ds.get_test_data(target='final')[0]
            return int(data_test.shape[0])
    except Exception:
        pass
    # Fall back to full combinatorial size when cached split is unavailable.
    return int(_count_total_exhaustive_samples(ds, nterms))

eval_mode = str(GENERALIZATION_EVAL_MODE).strip().lower()
if eval_mode not in {'sampled', 'exhaustive', 'hybrid'}:
    raise ValueError("GENERALIZATION_EVAL_MODE must be one of {'sampled', 'exhaustive', 'hybrid'}")

for n in eval_lengths:
    if n < 2:
        continue

    allowed, msg = enforce_length_constraint_for_model(config, n)
    if not allowed:
        row = {
            'nterms': int(n),
            'source': 'skipped',
            'in_distribution': (int(n) == base_nterms),
            'num_samples': 0,
            'final_accuracy': np.nan,
            'status': msg,
        }
        for i in range(1, int(n) + 1):
            row[f'term{i}_accuracy'] = np.nan
        generalization_rows.append(row)
        continue

    in_distribution = (int(n) == base_nterms)
    reused_stats = _reuse_in_distribution_stats_from_prefix_df(n) if in_distribution else None
    if reused_stats is not None:
        print(f'nterms={n}: reusing in-distribution metrics from prefix_df')
        stats = reused_stats
        eval_source = 'reused_prefix_df'
    else:
        use_exhaustive = (eval_mode == 'exhaustive') or (eval_mode == 'hybrid' and in_distribution)
        if use_exhaustive:
            estimated_total = _count_total_exhaustive_samples(dataset, n)
            print(f'nterms={n}: evaluating full exhaustive set in batches, total samples={estimated_total}')
            stats = evaluate_exhaustive_nterms_batched(
                model, dataset, nterms=n, batch_size=EVAL_BATCH_SIZE
            )
            eval_source = 'exhaustive_batched'
        else:
            dataset_size_n = _dataset_size_for_nterms(dataset, n)
            effective_samples = min(int(OOD_SAMPLES), int(dataset_size_n)) if int(dataset_size_n) > 0 else int(OOD_SAMPLES)
            sample_seed = int(OOD_SEED) + int(n)
            print(
                f'nterms={n}: evaluating sampled set, num_samples={effective_samples} '
                f'(min(OOD_SAMPLES={int(OOD_SAMPLES)}, dataset_size={int(dataset_size_n)})), seed={sample_seed}'
            )
            stats = evaluate_sampled_nterms(
                model,
                dataset,
                nterms=n,
                num_samples=effective_samples,
                seed=sample_seed,
                batch_size=EVAL_BATCH_SIZE,
            )
            eval_source = 'sampled'

    row = {
        'nterms': int(n),
        'source': eval_source,
        'in_distribution': in_distribution,
        'num_samples': int(stats['num_samples']),
        'final_accuracy': float(stats['final_accuracy']),
        'status': 'ok',
    }
    row.update(stats['per_term_accuracy'])
    generalization_rows.append(row)

if generalization_rows:
    generalization_df = pd.DataFrame(generalization_rows)
    term_cols = [f'term{i}_accuracy' for i in range(1, int(max_eval_nterms) + 1)]
    for c in term_cols:
        if c not in generalization_df.columns:
            generalization_df[c] = np.nan
    generalization_df = generalization_df[
        ['nterms', 'source', 'in_distribution', 'num_samples', 'final_accuracy'] + term_cols + ['status']
    ].sort_values(['nterms', 'in_distribution']).reset_index(drop=True)
else:
    term_cols = [f'term{i}_accuracy' for i in range(1, int(max_eval_nterms) + 1)]
    generalization_df = pd.DataFrame(columns=[
        'nterms',
        'source',
        'in_distribution',
        'num_samples',
        'final_accuracy',
        *term_cols,
        'status',
    ])

display(generalization_df)